# Configuration env

In [20]:
import pandas as pd

In [1]:
from pathlib import Path
import os

# remonte jusqu'au dossier qui contient .git, puis s'y place
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").is_dir())
os.chdir(ROOT)
print("racine projet :", ROOT)

racine projet : /Users/benjaminscemama/dev/market-risk-control


In [2]:
%load_ext sql
%config SqlMagic.autopandas = True
%sql duckdb:///:memory:
%sql ATTACH IF NOT EXISTS 'data/risk.db' AS r (TYPE sqlite);

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

,Success


In [3]:
%%sql
-- Test configuration environnement
SELECT name FROM (SHOW ALL TABLES) WHERE database = 'r' ORDER BY name;

Running query in 'duckdb:///:memory:'

,name
0,mkt_forward_curve
1,mkt_spot_hourly
2,pos_snapshot
3,ref_contract
4,ref_customer
5,ref_site
6,trd_deal


# 1. `ref_customer`

## Niveau 0 - cadrage

In [12]:
%%sql 
-- Nombre de lignes
select count(*) as nb_raw from r.ref_customer;

Running query in 'duckdb:///:memory:'

,nb_raw
0,220


---

In [36]:
%%sql
-- unicité de customer_id
select count(*) as n_raw, count(distinct customer_id) as n_distinct, count(customer_id) as n_customer_id from r.ref_customer;

Running query in 'duckdb:///:memory:'

,n_raw,n_distinct,n_customer_id
0,220,220,220


---

In [17]:
%%sql 
-- Unicité customer_name
select customer_name, count(*) as nb_customer_name from r.ref_customer
group by customer_name
having count(*) > 1;

Running query in 'duckdb:///:memory:'

,customer_name,nb_customer_name


In [44]:
%%sql 
select upper(trim(customer_name)) as customer_name from r.ref_customer
group by upper(trim(customer_name))
having count(*) > 1;

Running query in 'duckdb:///:memory:'

,customer_name


In [56]:
%%sql
with n as ( 
    select 
        customer_name as n0,
        upper(trim(customer_name)) as n1,
        replace(upper(trim(customer_name)), '.', '') as n2,
        regexp_replace(upper(trim(customer_name)), '\s+', ' ', 'g') as n3,
        regexp_replace(strip_accents(upper(trim(customer_name))), '[^A-Z0-9]', '', 'g') as n4
    from r.ref_customer
)
select 
    count(*) as lignes,
    count(distinct n0) as brut,
    count(distinct n1) as upper_trim,
    count(distinct n2) as sans_point,
    count(distinct n3) as espaces_normalises,
    count(distinct n4) as alphanumerique_seul
from n;


Running query in 'duckdb:///:memory:'

,lignes,brut,upper_trim,sans_point,espaces_normalises,alphanumerique_seul
0,220,220,220,220,220,220


---

In [23]:
%%sql
-- Exhaustivité « tout customer_id référencé dans ref_site ou ref_contract figure ici »
select rs.customer_id from r.ref_site as rs
left join r.ref_customer as rc on rs.customer_id = rc.customer_id
where rc.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id


In [25]:
%%sql
-- Exhaustivité « tout customer_id référencé dans ref_site ou ref_contract figure ici »
select rco.customer_id from r.ref_contract as rco
left join r.ref_customer as rc on rco.customer_id = rc.customer_id
where rc.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id


---

In [33]:
%%sql 
-- Tous les clients ont un site
select rc.customer_id from r.ref_customer as rc
left join r.ref_site as rs on rc.customer_id = rs.customer_id
where rs.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id


In [34]:
%%sql
-- Tous les clients ont un contrat
select rc.customer_id from r.ref_customer as rc
left join r.ref_contract as rco on rc.customer_id = rco.customer_id
where rco.customer_id is NULL;

Running query in 'duckdb:///:memory:'

,customer_id
0,C100035
1,C100132
2,C100046
3,C100103
4,C100116
...,...
69,C100152
70,C100194
71,C100013
72,C100076


---

# Niveau 1 - colonnes

In [27]:
df = %sql select * from r.ref_customer
df.nunique()

Running query in 'duckdb:///:memory:'

customer_id      220
customer_name    220
sector             7
segment            4
credit_rating      6
dtype: int64

In [30]:
df.isna().mean().sort_values(ascending=False)

customer_id      0.0
customer_name    0.0
sector           0.0
segment          0.0
credit_rating    0.0
dtype: float64

In [59]:
for c in ["sector", "segment", "credit_rating"] : 
    print(df[c].value_counts(dropna = False))

sector
Sante              39
Tertiaire          35
Distribution       33
Industrie          33
Collectivite       27
Transport          27
Agroalimentaire    26
Name: count, dtype: int64
segment
PME             74
ETI             66
PUBLIC          43
GRAND_COMPTE    37
Name: count, dtype: int64
credit_rating
BBB    64
BB     54
A      33
NR     31
B      27
AA     11
Name: count, dtype: int64


In [21]:
pd.crosstab(df.segment, df.sector)

sector,Agroalimentaire,Collectivite,Distribution,Industrie,Sante,Tertiaire,Transport
segment,,,,,,,
ETI,10,5,11,8,11,13,8
GRAND_COMPTE,4,3,6,8,6,5,5
PME,8,13,13,8,11,8,13
PUBLIC,4,6,3,9,11,9,1


# 2. `ref_site`

## Niveau 0 - cadrage

In [10]:
%%sql
select 
    count(*) as lignes,
    count(distinct site_id) as sites_distincts,
    count(distinct (site_id, commodity)) as site_commodity,
    count (distinct (site_id, commodity, dso)) as site_commodity_dso,
    count(distinct customer_id) as client
from r.ref_site;

Running query in 'duckdb:///:memory:'

,lignes,sites_distincts,site_commodity,site_commodity_dso,client
0,1400,1400,1400,1400,220


In [17]:
%%sql
select monitored, count(*) as n
from r.ref_site
group by monitored
order by monitored;

Running query in 'duckdb:///:memory:'

,monitored,n
0,0,900
1,1,500


In [47]:
df_ref_site = %sql select * from r.ref_site;
pd.crosstab(df_ref_site.dso, df_ref_site.region)

Running query in 'duckdb:///:memory:'

region,ARA,BRE,CVL,GES,HDF,IDF,NAQ,OCC,PACA,PDL
dso,,,,,,,,,,
ENEDIS,29,28,27,27,30,31,26,25,30,15
GEREDIS,37,37,26,36,33,23,28,24,24,29
GRDF,29,24,26,26,27,34,38,34,26,23
RESEAU_LOCAL,28,22,28,26,22,31,22,30,25,25
SRD,22,31,27,27,24,24,37,31,31,35


In [54]:
pd.crosstab(df_ref_site.dso, df_ref_site.commodity)

commodity,GAS,POWER
dso,,
ENEDIS,102,166
GEREDIS,109,188
GRDF,122,165
RESEAU_LOCAL,93,166
SRD,95,194


In [61]:
ko = df_ref_site.query("(dso == 'ENEDIS' and commodity == 'GAS') or (dso == 'GRDF' and commodity == 'POWER')")

print("lignes fautives :", len(ko), "sur", len(df_ref_site))
print("part des lignes :", len(ko) / len(df_ref_site))
print("puissance fautive :", ko.contracted_capacity_kw.sum(), "kW sur", df_ref_site.contracted_capacity_kw.sum())
print("part de la puissance :", ko.contracted_capacity_kw.sum() / df_ref_site.contracted_capacity_kw.sum())

lignes fautives : 267 sur 1400
part des lignes : 0.19071428571428573
puissance fautive : 1125523.0 kW sur 5242077.0
part de la puissance : 0.21470936043098948
